In [ ]:
import os
from pathlib import Path

# Nếu chạy trên Kaggle, clone repo vào /kaggle/working/PRISM (chỉ khi chưa có repo)
if os.path.exists('/kaggle/working'):
    REPO_DIR = '/kaggle/working/PRISM'
    repo_path = Path(REPO_DIR)
    # Clone nếu thư mục chưa tồn tại hoặc không phải repo git
    if not repo_path.exists() or not (repo_path / '.git').exists():
        # Nếu thư mục tồn tại nhưng không rỗng, tránh ghi đè
        if repo_path.exists() and any(repo_path.iterdir()):
            print(f"{REPO_DIR} tồn tại và không rỗng — bỏ qua git clone.")
        else:
            print('Cloning PRISM into', REPO_DIR)
            get_ipython().system(f'git clone https://github.com/ptdbrain/PRISM.git {REPO_DIR}')
    else:
        print('Repository already present at', REPO_DIR)
else:
    # Không phải Kaggle — thử Colab mount hoặc dùng cwd
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=True)
        REPO_DIR = "/content/drive/Othercomputers/MyLaptop/Quantization/PRISM"
    except Exception:
        REPO_DIR = os.path.abspath('.')

# Chuyển vào thư mục làm việc (sẽ tồn tại sau khi clone)
%cd $REPO_DIR
print('REPO_DIR:', REPO_DIR)


Mounted at /content/drive


In [ ]:
# REPO_DIR is set in the first cell; no manual cd needed here.
print("Current working directory:", REPO_DIR)
!pwd


In [ ]:
!pip install -r {REPO_DIR}/requirements.txt


In [ ]:
# Cell 1: setup
# REPO_DIR is defined in the first cell; if not, set a sane default.
REPO_DIR = globals().get('REPO_DIR', '/kaggle/working/PRISM')

# If you need to clone the repository, uncomment and adjust:
# %cd /content
# !git clone <REPO_URL> PRISM

%cd $REPO_DIR
!pip install -q -U pip
!pip install -q -e . sentencepiece accelerate

import torch

print("CUDA:", torch.cuda.is_available())
GPU_COUNT = torch.cuda.device_count()
print("GPU count:", GPU_COUNT)
for gpu_id in range(GPU_COUNT):
    props = torch.cuda.get_device_properties(gpu_id)
    total_gib = props.total_memory / (1024 ** 3)
    print(f"GPU {gpu_id}: {torch.cuda.get_device_name(gpu_id)} | {total_gib:.1f} GiB")


In [ ]:
# Cài đặt ninja để hỗ trợ JIT và đồng bộ lại các thư viện vision/audio với PyTorch 2.11.0 (CUDA 13.0)
!pip install -q ninja
# !pip install --force-reinstall -q torch==2.11.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu130
print("\nCài đặt hoàn tất. QUAN TRỌNG: Hãy chọn 'Runtime' -> 'Restart Session' trước khi chạy lại bước profiling.")


Cài đặt hoàn tất. QUAN TRỌNG: Hãy chọn 'Runtime' -> 'Restart Session' trước khi chạy lại bước profiling.


In [ ]:
from __future__ import annotations

from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import gc
import json
import os
import subprocess
import sys

import torch

MODEL_NAME = "Qwen/Qwen2.5-1.5B"
NUM_SHARDS = 16
START_SHARD_ID = 0
GROUP_SIZE = 128
PRISM_BITS = (2, 3, 4)
TRUST_REMOTE_CODE = False
TORCH_DTYPE = torch.float16
GPU_RESERVE_GIB = 1.5

sanitized_model_name = MODEL_NAME.replace("/", "_")
OUT_DIR = Path(f"{REPO_DIR}/artifacts/prism/{sanitized_model_name}_meta")
OUT_DIR.mkdir(parents=True, exist_ok=True)
WORKER_SCRIPT = Path(f"{REPO_DIR}/_multi_gpu_stage_worker.py")
if not WORKER_SCRIPT.exists():
    raise FileNotFoundError(WORKER_SCRIPT)


def gpu_inventory() -> list[dict[str, object]]:
    inventory = []
    for gpu_id in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(gpu_id)
        inventory.append({
            "id": gpu_id,
            "name": torch.cuda.get_device_name(gpu_id),
            "total_gib": props.total_memory / (1024 ** 3),
        })
    return inventory


def dtype_nbytes(dtype: torch.dtype) -> int:
    if dtype in (torch.float16, torch.bfloat16):
        return 2
    if dtype == torch.float32:
        return 4
    return torch.tensor([], dtype=dtype).element_size()


def estimate_model_weight_size_gib(
    model_name: str,
    torch_dtype: torch.dtype = torch.float16,
    trust_remote_code: bool = False,
) -> float:
    from accelerate import init_empty_weights
    from transformers import AutoConfig, AutoModelForCausalLM

    config = AutoConfig.from_pretrained(model_name, trust_remote_code=trust_remote_code)
    with init_empty_weights():
        model = AutoModelForCausalLM.from_config(config, trust_remote_code=trust_remote_code)
    total_params = sum(param.numel() for param in model.parameters())
    del model
    return total_params * dtype_nbytes(torch_dtype) / (1024 ** 3)


def single_gpu_usable_gib(gpu_id: int = 0, reserve_gib: float = GPU_RESERVE_GIB) -> float:
    total_gib = torch.cuda.get_device_properties(gpu_id).total_memory / (1024 ** 3)
    return max(1.0, total_gib - reserve_gib)


def probe_single_gpu_load(
    model_name: str,
    torch_dtype: torch.dtype = torch.float16,
    trust_remote_code: bool = False,
    gpu_id: int = 0,
) -> tuple[bool, str | None]:
    from transformers import AutoModelForCausalLM

    usable_gib = int(single_gpu_usable_gib(gpu_id))
    try:
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch_dtype,
            trust_remote_code=trust_remote_code,
            low_cpu_mem_usage=True,
            device_map={"": gpu_id},
            max_memory={gpu_id: f"{usable_gib}GiB"},
        )
        model.eval()
        del model
        gc.collect()
        torch.cuda.empty_cache()
        return True, None
    except RuntimeError as exc:
        gc.collect()
        torch.cuda.empty_cache()
        message = str(exc)
        if "out of memory" in message.lower() or "cuda" in message.lower():
            return False, message
        raise


def choose_execution_mode(
    model_name: str,
    torch_dtype: torch.dtype = torch.float16,
    trust_remote_code: bool = False,
) -> dict[str, object]:
    inventory = gpu_inventory()
    mode_info: dict[str, object] = {
        "gpu_count": len(inventory),
        "gpus": inventory,
        "estimated_weight_gib": None,
        "single_gpu_usable_gib": None,
        "mode": "single_gpu",
        "reason": "Only one CUDA device is visible.",
    }
    if len(inventory) < 2:
        return mode_info

    estimated_weight_gib = estimate_model_weight_size_gib(
        model_name,
        torch_dtype=torch_dtype,
        trust_remote_code=trust_remote_code,
    )
    single_gpu_budget = min(single_gpu_usable_gib(0), single_gpu_usable_gib(1))
    mode_info["estimated_weight_gib"] = round(estimated_weight_gib, 3)
    mode_info["single_gpu_usable_gib"] = round(single_gpu_budget, 3)

    if estimated_weight_gib <= single_gpu_budget * 0.88:
        fits, error = probe_single_gpu_load(
            model_name,
            torch_dtype=torch_dtype,
            trust_remote_code=trust_remote_code,
            gpu_id=0,
        )
        if fits:
            mode_info["mode"] = "task_parallel"
            mode_info["reason"] = (
                "Estimated weights fit in one T4, so split independent PRISM work "
                "across both GPUs instead of using redundant data parallelism."
            )
            return mode_info
        mode_info["probe_error"] = error

    mode_info["mode"] = "model_parallel"
    mode_info["reason"] = (
        "Single-GPU fit estimate or probe failed, so load the model across both T4s "
        "with device_map='auto'."
    )
    return mode_info


def split_balanced(items: list[int], num_parts: int) -> list[list[int]]:
    parts = [[] for _ in range(num_parts)]
    loads = [0 for _ in range(num_parts)]
    for item in items:
        target = min(range(num_parts), key=lambda idx: loads[idx])
        parts[target].append(item)
        loads[target] += 1
    return parts


def stream_command(cmd: list[str], gpu_id: int) -> int:
    env = dict(os.environ)
    env.setdefault("TOKENIZERS_PARALLELISM", "false")
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=env,
    )
    assert proc.stdout is not None
    print(f"\n{'=' * 20} GPU {gpu_id} {'=' * 20}")
    for line in proc.stdout:
        print(line.rstrip())
    return proc.wait()


def run_sensitivity_dataset_task_parallel(mode_info: dict[str, object]) -> None:
    remaining = list(range(START_SHARD_ID, NUM_SHARDS))
    remaining = [
        shard_id
        for shard_id in remaining
        if not (OUT_DIR / f"train_shard_{shard_id:02d}_of_{NUM_SHARDS}.pt").exists()
    ]
    print(f"Shards to process: {remaining} ({len(remaining)} total)")
    if not remaining:
        print("All shards already computed!")
        return

    worker_count = min(2, int(mode_info["gpu_count"]))
    assignments = split_balanced(remaining, worker_count)
    commands: list[tuple[int, list[str]]] = []
    for gpu_id, shard_ids in enumerate(assignments):
        if not shard_ids:
            continue
        commands.append((
            gpu_id,
            [
                sys.executable,
                str(WORKER_SCRIPT),
                "--stage", "sensitivity_dataset",
                "--gpu", str(gpu_id),
                "--shards", ",".join(str(shard_id) for shard_id in shard_ids),
                "--num-shards", str(NUM_SHARDS),
                "--model", MODEL_NAME,
                "--repo-dir", REPO_DIR,
                "--out-dir", str(OUT_DIR),
                "--group-size", str(GROUP_SIZE),
                "--bits", ",".join(str(bit) for bit in PRISM_BITS),
                "--torch-dtype", "float16",
                "--n-samples", "16",
                "--seq-len", "512",
            ],
        ))

    failures = []
    with ThreadPoolExecutor(max_workers=len(commands)) as executor:
        future_map = {
            executor.submit(stream_command, cmd, gpu_id): gpu_id
            for gpu_id, cmd in commands
        }
        for future in as_completed(future_map):
            gpu_id = future_map[future]
            return_code = future.result()
            if return_code != 0:
                failures.append((gpu_id, return_code))

    if failures:
        raise RuntimeError(f"Shard workers failed: {failures}")
    print(f"Done! Check {OUT_DIR}")


MODE_INFO = choose_execution_mode(
    MODEL_NAME,
    torch_dtype=TORCH_DTYPE,
    trust_remote_code=TRUST_REMOTE_CODE,
)
print(json.dumps(MODE_INFO, indent=2))

if MODE_INFO["mode"] == "task_parallel":
    run_sensitivity_dataset_task_parallel(MODE_INFO)
else:
    print(
        "Model-parallel mode selected. Skip shard generation here and use the helper cell below "
        "to run Stage 1-4 with device_map='auto'."
    )


In [ ]:
from __future__ import annotations

from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import json
import tempfile

import torch

from prism.assign.optimize import assign_bits
from prism.data.schemas import ProfileArtifact
from prism.profile.inspect import iter_named_linear_layers
from prism.profile.pipeline import build_profile_artifact
from prism.rtn.precompute import precompute_model_rtn
from prism.runtime.assemble import assemble_runtime_model


def load_model_single_gpu(model_name: str, torch_dtype: torch.dtype = torch.float16, trust_remote_code: bool = False):
    from transformers import AutoModelForCausalLM

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch_dtype,
        trust_remote_code=trust_remote_code,
        low_cpu_mem_usage=True,
        device_map={"": 0},
        max_memory={0: f"{int(single_gpu_usable_gib(0))}GiB"},
    )
    model.eval()
    return model


def load_model_model_parallel(model_name: str, torch_dtype: torch.dtype = torch.float16, trust_remote_code: bool = False):
    from transformers import AutoModelForCausalLM

    max_memory = {gpu_id: f"{int(single_gpu_usable_gib(gpu_id))}GiB" for gpu_id in range(min(2, torch.cuda.device_count()))}
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch_dtype,
        trust_remote_code=trust_remote_code,
        low_cpu_mem_usage=True,
        device_map="auto",
        max_memory=max_memory,
    )
    model.eval()
    return model


def layer_param_counts(model) -> dict[str, int]:
    return {name: int(module.weight.numel()) for name, module in iter_named_linear_layers(model)}


def split_layers_balanced(layer_sizes: dict[str, int], num_parts: int) -> list[list[str]]:
    parts = [[] for _ in range(num_parts)]
    loads = [0 for _ in range(num_parts)]
    for layer_name, size in sorted(layer_sizes.items(), key=lambda item: item[1], reverse=True):
        target = min(range(num_parts), key=lambda idx: loads[idx])
        parts[target].append(layer_name)
        loads[target] += size
    return parts


def _write_layer_list(layer_names: list[str], directory: Path, suffix: str) -> Path:
    path = directory / f"layers_{suffix}.json"
    path.write_text(json.dumps(layer_names), encoding="utf-8")
    return path


def _launch_stage_commands(commands: list[tuple[int, list[str]]]) -> None:
    failures = []
    with ThreadPoolExecutor(max_workers=len(commands)) as executor:
        future_map = {
            executor.submit(stream_command, cmd, gpu_id): gpu_id
            for gpu_id, cmd in commands
        }
        for future in as_completed(future_map):
            gpu_id = future_map[future]
            return_code = future.result()
            if return_code != 0:
                failures.append((gpu_id, return_code))
    if failures:
        raise RuntimeError(f"Stage workers failed: {failures}")


def merge_profile_artifacts(part_paths: list[Path], merged_path: Path) -> ProfileArtifact:
    merged_layers = []
    metadata = None
    model_id = None
    model_family = None
    seen = set()
    for path in part_paths:
        payload = json.loads(path.read_text(encoding="utf-8"))
        metadata = metadata or payload.get("metadata", {})
        model_id = model_id or payload.get("model_id", MODEL_NAME)
        model_family = model_family or payload.get("model_family", "unknown")
        for layer in payload["layers"]:
            if layer["layer_name"] not in seen:
                merged_layers.append(layer)
                seen.add(layer["layer_name"])
    artifact = ProfileArtifact.from_dict({
        "model_id": model_id,
        "model_family": model_family,
        "layers": merged_layers,
        "metadata": metadata or {},
    })
    merged_path.write_text(json.dumps(artifact.to_dict(), indent=2), encoding="utf-8")
    return artifact


def merge_rtn_manifests(manifest_paths: list[Path], merged_dir: Path) -> dict[str, object]:
    merged_dir.mkdir(parents=True, exist_ok=True)
    merged = None
    for manifest_path in manifest_paths:
        payload = json.loads(manifest_path.read_text(encoding="utf-8"))
        if merged is None:
            merged = payload
            merged["artifact_root"] = str(merged_dir)
            merged["layers"] = dict(payload["layers"])
        else:
            merged["layers"].update(payload["layers"])
    manifest_path = merged_dir / "manifest.json"
    manifest_path.write_text(json.dumps(merged, indent=2), encoding="utf-8")
    return merged


def run_profile_artifact_task_parallel(
    model_name: str,
    mlp_path: str | Path,
    output_dir: Path,
    torch_dtype: torch.dtype = torch.float16,
    trust_remote_code: bool = False,
    group_size: int = GROUP_SIZE,
) -> ProfileArtifact:
    planning_model = load_model_single_gpu(model_name, torch_dtype=torch_dtype, trust_remote_code=trust_remote_code)
    layer_splits = split_layers_balanced(layer_param_counts(planning_model), num_parts=2)
    del planning_model
    torch.cuda.empty_cache()

    output_dir.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(dir=output_dir) as tmp_dir:
        tmp_dir_path = Path(tmp_dir)
        commands = []
        for gpu_id, layer_names in enumerate(layer_splits):
            layer_json = _write_layer_list(layer_names, tmp_dir_path, f"profile_gpu{gpu_id}")
            commands.append((
                gpu_id,
                [
                    sys.executable,
                    str(WORKER_SCRIPT),
                    "--stage", "profile_artifact",
                    "--gpu", str(gpu_id),
                    "--model", model_name,
                    "--repo-dir", REPO_DIR,
                    "--out-dir", str(output_dir),
                    "--group-size", str(group_size),
                    "--torch-dtype", "float16",
                    "--mlp-path", str(mlp_path),
                    "--layer-names-json", str(layer_json),
                ],
            ))
        _launch_stage_commands(commands)

    part_paths = [output_dir / f"profile_gpu{gpu_id}.json" for gpu_id in range(2)]
    return merge_profile_artifacts(part_paths, output_dir / "profile.json")


def run_rtn_precompute_task_parallel(
    model_name: str,
    output_dir: Path,
    torch_dtype: torch.dtype = torch.float16,
    trust_remote_code: bool = False,
    group_size: int = GROUP_SIZE,
    bits: tuple[int, ...] = PRISM_BITS,
) -> dict[str, object]:
    planning_model = load_model_single_gpu(model_name, torch_dtype=torch_dtype, trust_remote_code=trust_remote_code)
    layer_splits = split_layers_balanced(layer_param_counts(planning_model), num_parts=2)
    del planning_model
    torch.cuda.empty_cache()

    output_dir.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(dir=output_dir) as tmp_dir:
        tmp_dir_path = Path(tmp_dir)
        commands = []
        for gpu_id, layer_names in enumerate(layer_splits):
            layer_json = _write_layer_list(layer_names, tmp_dir_path, f"rtn_gpu{gpu_id}")
            commands.append((
                gpu_id,
                [
                    sys.executable,
                    str(WORKER_SCRIPT),
                    "--stage", "rtn_precompute",
                    "--gpu", str(gpu_id),
                    "--model", model_name,
                    "--repo-dir", REPO_DIR,
                    "--out-dir", str(output_dir),
                    "--group-size", str(group_size),
                    "--bits", ",".join(str(bit) for bit in bits),
                    "--torch-dtype", "float16",
                    "--layer-names-json", str(layer_json),
                ],
            ))
        _launch_stage_commands(commands)

    manifest_paths = [output_dir / f"gpu{gpu_id}" / "manifest.json" for gpu_id in range(2)]
    return merge_rtn_manifests(manifest_paths, output_dir)


def run_prism_pipeline_kaggle(
    model_name: str,
    mlp_path: str | Path,
    target_average_bits: float = 3.0,
    torch_dtype: torch.dtype = torch.float16,
    trust_remote_code: bool = False,
    group_size: int = GROUP_SIZE,
):
    artifacts_root = Path(REPO_DIR) / "artifacts" / "prism" / model_name.replace("/", "_")
    artifacts_root.mkdir(parents=True, exist_ok=True)
    mode_info = choose_execution_mode(model_name, torch_dtype=torch_dtype, trust_remote_code=trust_remote_code)
    print(json.dumps(mode_info, indent=2))

    if mode_info["mode"] == "model_parallel":
        base_model = load_model_model_parallel(model_name, torch_dtype=torch_dtype, trust_remote_code=trust_remote_code)
        profile = build_profile_artifact(
            base_model,
            mlp_path=Path(mlp_path),
            output_path=artifacts_root / "profile.json",
            model_id=model_name,
            model_family="unknown",
            group_size=group_size,
        )
        assignment = assign_bits(profile, target_average_bits=target_average_bits)
        manifest = precompute_model_rtn(base_model, output_dir=artifacts_root / "rtn", group_size=group_size, bits=PRISM_BITS)
        runtime_model, runtime_summary = assemble_runtime_model(
            base_model,
            manifest,
            assignment,
            artifact_root=artifacts_root / "rtn",
            copy_model=False,
        )
    else:
        profile = run_profile_artifact_task_parallel(
            model_name,
            mlp_path=mlp_path,
            output_dir=artifacts_root / "profile_parts",
            torch_dtype=torch_dtype,
            trust_remote_code=trust_remote_code,
            group_size=group_size,
        )
        assignment = assign_bits(profile, target_average_bits=target_average_bits)
        manifest = run_rtn_precompute_task_parallel(
            model_name,
            output_dir=artifacts_root / "rtn",
            torch_dtype=torch_dtype,
            trust_remote_code=trust_remote_code,
            group_size=group_size,
            bits=PRISM_BITS,
        )
        base_model = load_model_single_gpu(model_name, torch_dtype=torch_dtype, trust_remote_code=trust_remote_code)
        runtime_model, runtime_summary = assemble_runtime_model(
            base_model,
            manifest,
            assignment,
            artifact_root=artifacts_root / "rtn",
            copy_model=True,
        )

    return {
        "mode_info": mode_info,
        "profile": profile,
        "assignment": assignment,
        "manifest": manifest,
        "runtime_model": runtime_model,
        "runtime_summary": runtime_summary,
    }


# Example, only after `prism_mlp.pt` exists:
# result = run_prism_pipeline_kaggle(
#     model_name=MODEL_NAME,
#     mlp_path=Path(REPO_DIR) / "artifacts" / "prism" / f"{sanitized_model_name}_merged" / "prism_mlp.pt",
#     target_average_bits=3.0,
# )


In [ ]:
from pathlib import Path
import torch
from prism.profiling.meta_learner import train_meta_learner

# REPO_DIR and MODEL_NAME are assumed to be available from previous cells.

# Sanitize MODEL_NAME for directory creation
sanitized_model_name = MODEL_NAME.replace("/", "_")

# Thư mục mới để lưu dữ liệu đã gộp và MLP đã huấn luyện
out_dir = Path(f"{REPO_DIR}/artifacts/prism/{sanitized_model_name}_merged")
out_dir.mkdir(parents=True, exist_ok=True)

shard_paths = sorted(Path(f"{REPO_DIR}/artifacts/prism/{sanitized_model_name}_meta").glob("train_shard_*.pt"))

Xs, Ys, metas = [], [], []
feature_order = None

for p in shard_paths:
    b = torch.load(p, map_location="cpu", weights_only=False)
    Xs.append(b["X"])
    Ys.append(b["Y"])
    metas.extend(b["meta"])
    feature_order = b.get("feature_order", feature_order)

bundle = {
    "X": torch.cat(Xs, dim=0),
    "Y": torch.cat(Ys, dim=0),
    "meta": metas,
    "feature_order": feature_order,
}

merged_path = out_dir / "prism_train_data.pt"
torch.save(bundle, merged_path)

train_meta_learner(
    dataset_path=str(merged_path),
    epochs=100,
    save_path=str(out_dir / "prism_mlp.pt"),
)

print("records:", bundle["X"].shape[0])
print("saved:", out_dir / "prism_mlp.pt")

In [ ]:
import torch
from pathlib import Path

# REPO_DIR and MODEL_NAME are assumed to be available from previous cells.
sanitized_model_name = MODEL_NAME.replace("/", "_")
base_path = Path(f"{REPO_DIR}/artifacts/prism/{sanitized_model_name}_merged")

bundle = torch.load(
    base_path / "prism_train_data.pt",
    map_location="cpu",
    weights_only=False,
)

print("X shape:", bundle["X"].shape)
print("Y shape:", bundle["Y"].shape)
print("num rows:", len(bundle["meta"]))
print("first rows:", bundle["meta"][:3])
print(f"MLP saved at: {base_path}/prism_mlp.pt")

In [ ]:
from pathlib import Path

# REPO_DIR and MODEL_NAME are assumed to be available from previous cells.
sanitized_model_name = MODEL_NAME.replace("/", "_")
target_dir = f"{REPO_DIR}/artifacts/prism/{sanitized_model_name}_merged"

!pwd
!ls -lah {target_dir}
!find . -maxdepth 5 -type f | grep "artifacts/prism/{sanitized_model_name}_merged/prism_train_data.pt\|artifacts/prism/{sanitized_model_name}_merged/prism_mlp.pt"